# Exercice 02 – Corrigé

Version corrigée du notebook `02_chat_boucle.ipynb`.

In [ ]:
import json
import time

import requests
from openai import OpenAI

In [ ]:
BASE_URL = "http://127.0.0.1:8080"
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="pas-besoin")

SYSTEM_PROMPT = "Tu es un assistant concis et sympathique. Tu réponds en français."

print(requests.get(f"{BASE_URL}/health").json())

## Étape 1 – Taille du contexte

In [ ]:
props = requests.get(f"{BASE_URL}/props").json()
N_CTX = props["default_generation_settings"]["n_ctx"]
print("Contexte par slot :", N_CTX)

## Étape 2 – Réponse en streaming

In [ ]:
def repondre(history: list) -> tuple[str, dict]:
    """Affiche la réponse en streaming, renvoie (texte, stats)."""
    debut = time.perf_counter()
    stream = client.chat.completions.create(
        model="local",
        messages=history,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

    buffer = ""
    stats = {}
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            delta = chunk.choices[0].delta.content
            print(delta, end="", flush=True)
            buffer += delta
        if chunk.usage:                              # dernier chunk
            stats["prompt_tokens"] = chunk.usage.prompt_tokens
            stats["completion_tokens"] = chunk.usage.completion_tokens
            timings = chunk.model_extra.get("timings")
            if timings:
                stats["tok_par_s"] = timings["predicted_per_second"]

    if "tok_par_s" not in stats and "completion_tokens" in stats:   # secours
        stats["tok_par_s"] = stats["completion_tokens"] / (time.perf_counter() - debut)

    print()
    return buffer, stats

_ = repondre([{"role": "user", "content": "Dis bonjour en une phrase."}])

## Étape 3 – Boucle de chat (version de base)

In [ ]:
def chat():
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    dernier_prompt_tokens = 0

    while True:
        saisie = input("Vous > ").strip()

        if saisie.lower() == "quit":
            print("Au revoir.")
            break

        if saisie == "/reset":
            history = [{"role": "system", "content": SYSTEM_PROMPT}]
            dernier_prompt_tokens = 0
            print("Historique vidé.")
            continue

        if saisie == "/history":
            print(f"{len(history)} messages, dernier prompt : {dernier_prompt_tokens} tokens / {N_CTX}")
            continue

        if not saisie:
            continue

        history.append({"role": "user", "content": saisie})
        print("Bot  > ", end="")
        reponse, stats = repondre(history)
        history.append({"role": "assistant", "content": reponse})   # rôle assistant : ce sont SES paroles

        dernier_prompt_tokens = stats.get("prompt_tokens", 0)
        print(f"       [{stats.get('completion_tokens', '?')} tokens, {stats.get('tok_par_s', 0):.1f} tok/s]")

    return history

history = chat()

## Étape 4 – Alerte et fenêtre glissante

In [ ]:
SEUIL = 0.8

def elaguer(history: list, prompt_tokens: int) -> list:
    """Supprime les plus vieux couples user/assistant tant que nous dépassons le seuil.
    Approximation : nous retirons des messages et nous estimons les tokens économisés
    par une règle simple (1 token ≈ 4 caractères)."""
    while prompt_tokens > SEUIL * N_CTX and len(history) > 3:
        supprimes = history[1:3]                    # le plus vieux couple, le system reste en 0
        history = [history[0]] + history[3:]
        prompt_tokens -= sum(len(m["content"]) for m in supprimes) // 4
        print(f"       [fenêtre glissante : 2 messages supprimés, il en reste {len(history)}]")
    return history


def chat_v2():
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    dernier_prompt_tokens = 0

    while True:
        saisie = input("Vous > ").strip()

        if saisie.lower() == "quit":
            break
        if saisie == "/reset":
            history = [{"role": "system", "content": SYSTEM_PROMPT}]
            dernier_prompt_tokens = 0
            print("Historique vidé.")
            continue
        if saisie == "/history":
            print(f"{len(history)} messages, dernier prompt : {dernier_prompt_tokens} tokens / {N_CTX}")
            continue
        if saisie.startswith("/save "):
            chemin = saisie.split(maxsplit=1)[1]
            with open(chemin, "w", encoding="utf-8") as f:
                json.dump(history, f, ensure_ascii=False, indent=2)
            print(f"Historique sauvegardé dans {chemin}")
            continue
        if not saisie:
            continue

        history.append({"role": "user", "content": saisie})
        print("Bot  > ", end="")
        reponse, stats = repondre(history)
        history.append({"role": "assistant", "content": reponse})

        dernier_prompt_tokens = stats.get("prompt_tokens", 0)
        print(f"       [{stats.get('completion_tokens', '?')} tokens, {stats.get('tok_par_s', 0):.1f} tok/s, "
              f"prompt {dernier_prompt_tokens}/{N_CTX}]")

        if dernier_prompt_tokens > SEUIL * N_CTX:
            print(f"       ⚠ Contexte à plus de {int(SEUIL*100)} % !")
            history = elaguer(history, dernier_prompt_tokens)

    return history

history = chat_v2()

## Éléments de réponse aux questions de réflexion

1. **`assistant` et pas `system`** : le rôle indique *qui parle*. Le template entoure les messages `assistant` des marqueurs que le modèle associe à ses propres paroles ; il « se souvient » ainsi de ce qu'il a dit. En `system`, le texte serait lu comme une consigne, et la plupart des templates n'acceptent qu'un seul bloc system au début.
2. **Oubli de la réponse dans l'historique** : le modèle voit une suite de questions utilisateur sans aucune réponse. Il répond souvent à la première question, ou mélange tout, et ne peut plus se référer à ce qu'il a dit.
3. **Le tour 10 n'est pas 10× plus lent** : `llama-server` garde le KV cache du préfixe commun (`cache_prompt`, actif par défaut). `usage.prompt_tokens` grossit à chaque tour mais `timings["prompt_n"]`, le nombre de tokens réellement traités, reste petit.
4. **Inconvénients de la fenêtre glissante** : nous perdons les informations du début (le prénom, les consignes données en cours de route). Alternatives : demander au modèle un **résumé** des anciens tours et le remplacer dans l'historique ; garder les *n* premiers tours en plus des *n* derniers ; ou une mémoire externe (recherche dans les anciens messages).